# Notebook 01 — Laplace Solver (2D Surface Trap)

This notebook builds a simple 2D finite-difference Laplace solver for a surface-style electrode layout.

It is intended as a lightweight starting point for **Ion Trap Parameter Lab**:

- define a rectangular computational grid
- place simple electrode regions on the lower boundary
- solve Laplace's equation with fixed-potential boundary conditions
- visualize the potential and a vertical centerline cut

This is a first electrostatics notebook, not yet a full RF pseudopotential or full trap optimization model.


## Imports


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve


## Grid and geometry

We use a rectangular domain with Dirichlet boundaries.

A simple surface-style electrode pattern is imposed along the lower boundary:

- left DC electrode: negative voltage
- center RF electrode: positive voltage
- right DC electrode: negative voltage

The remaining outer boundaries are grounded.


In [ ]:
# Domain size and resolution
nx, ny = 121, 81
x = np.linspace(-3.0, 3.0, nx)
y = np.linspace(0.0, 4.0, ny)
dx = x[1] - x[0]
dy = y[1] - y[0]

# Potential array and mask for fixed-potential nodes
V = np.zeros((ny, nx), dtype=float)
fixed = np.zeros((ny, nx), dtype=bool)

# Ground outer boundaries
V[:, 0] = 0.0
V[:, -1] = 0.0
V[-1, :] = 0.0
fixed[:, 0] = True
fixed[:, -1] = True
fixed[-1, :] = True

# Lower boundary electrodes
bottom = 0

left_dc = (x >= -2.2) & (x <= -0.9)
center_rf = (x >= -0.5) & (x <= 0.5)
right_dc = (x >= 0.9) & (x <= 2.2)
remaining_ground = ~(left_dc | center_rf | right_dc)

V[bottom, left_dc] = -1.0
V[bottom, center_rf] = 1.0
V[bottom, right_dc] = -1.0
V[bottom, remaining_ground] = 0.0
fixed[bottom, :] = True

print(f'Grid: {nx} x {ny}')
print(f'dx = {dx:.3f}, dy = {dy:.3f}')


## Solve Laplace equation

We solve

$$
\nabla^2 V = 0
$$

using a standard 5-point finite-difference stencil and a sparse linear solve.


In [ ]:
def idx(i, j, nx):
    return i * nx + j

N = nx * ny
A = lil_matrix((N, N), dtype=float)
b = np.zeros(N, dtype=float)

cx = 1.0 / dx**2
cy = 1.0 / dy**2
c0 = -2.0 * (cx + cy)

for i in range(ny):
    for j in range(nx):
        k = idx(i, j, nx)
        
        if fixed[i, j]:
            A[k, k] = 1.0
            b[k] = V[i, j]
        else:
            A[k, idx(i, j, nx)] = c0
            A[k, idx(i, j - 1, nx)] = cx
            A[k, idx(i, j + 1, nx)] = cx
            A[k, idx(i - 1, j, nx)] = cy
            A[k, idx(i + 1, j, nx)] = cy

solution = spsolve(A.tocsr(), b)
Vsol = solution.reshape((ny, nx))

print('Laplace solve complete.')
print(f'Potential range: [{Vsol.min():.3f}, {Vsol.max():.3f}]')


## Plot geometry and potential


In [ ]:
X, Y = np.meshgrid(x, y)

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.pcolormesh(X, Y, Vsol, shading='auto')
cs = ax.contour(X, Y, Vsol, levels=15)
ax.clabel(cs, inline=True, fontsize=8)

# electrode guide bars on lower boundary
ax.plot(x[left_dc], np.full(np.sum(left_dc), y[0]), linewidth=6)
ax.plot(x[center_rf], np.full(np.sum(center_rf), y[0]), linewidth=6)
ax.plot(x[right_dc], np.full(np.sum(right_dc), y[0]), linewidth=6)

ax.set_title('2D Electrostatic Potential for a Simple Surface Electrode Layout')
ax.set_xlabel('x')
ax.set_ylabel('y')
fig.colorbar(im, ax=ax, label='Potential')
plt.tight_layout()
plt.show()


## Vertical centerline cut

A quick diagnostic is the potential along the vertical centerline. This helps show how the boundary electrodes shape the potential above the surface.


In [ ]:
j0 = np.argmin(np.abs(x - 0.0))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(y, Vsol[:, j0])
ax.set_title('Potential Along the Vertical Centerline (x = 0)')
ax.set_xlabel('y')
ax.set_ylabel('Potential')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Notes and next steps

This notebook is intentionally minimal. Good next extensions:

- compute the electric field $\mathbf{E} = -\nabla V$
- vary electrode widths and gaps
- add parameter sweeps over applied voltages
- build RF pseudopotential approximations
- extract curvature and approximate confinement metrics

That sequence will move this from a basic electrostatics notebook toward a practical ion-trap design workflow.
